# Fine-tuned heads for all sixteen stations (2026-08-18)

The only test this project trusts is what the scanner does on raw audio, so the
fine-tuned arm needs heads for every station, not three. This produces them.

Two things differ from the earlier notebooks.

**All background is used.** Background is no longer withheld by station.
Collecting negatives from a site's own rejected detections is part of the method
-- deploy, a reviewer rejects the false positives, they become negatives -- so a
fold that withholds them measures a situation no user is in. Confirmed calls
stay withheld, because those need an expert and a new site does not have them.
The figure this produces is therefore the value of one round of
review-and-retrain at a station, **not** transfer to an unheard station, and
must not be reported as a held-out precision.

**Folds run in batches of four and sync to Drive after each batch.** Sixteen
fine-tuned folds is roughly eight hours on a T4, which is longer than a session
can be relied on. A disconnect after batch three should cost the fourth batch,
not all four.


In [ ]:
from google.colab import drive; drive.mount('/content/drive')
!rm -rf /content/repo && git clone -q -b v13-honest-labels https://github.com/Mo119m/primates-sound-detection /content/repo
%cd /content/repo
!git log --oneline -1
U = '/content/drive/MyDrive/primates-sound-detection/upload_to_drive_CLEAN'
!mkdir -p /content/dataC && cp -n {U}/v13_images.npy {U}/v13_index.csv {U}/manifest.csv /content/dataC/
!ls -la /content/dataC


In [ ]:
A = '--manifest /content/dataC/manifest.csv --index /content/dataC/v13_index.csv --images /content/dataC/v13_images.npy --cache /content/dataC/v13_features.npy'
!python scripts/train_v13_loso.py --prepare-cache-only --overwrite {A} --out /content/warm.csv --run-metadata /content/cacheC.run.json


In [ ]:
import os, shutil, subprocess
OUT = '/content/drive/MyDrive/primates-sound-detection/colab_b34_16'
os.makedirs(OUT, exist_ok=True)
HEADS = '/content/heads_b34_16'
os.makedirs(HEADS, exist_ok=True)
SEP = '===== batch'

BATCHES = [
    'IPA1ST,IPA2ST,IPA4ST,IPA6ST',
    'IPA7ST,IPA8ST,IPA10ST,IPA11ST',
    'IPA13ST,IPA14ST,IPA15ST,IPA16ST',
    'IPA17ST,IPA18ST,IPA19ST,IPA20ST',
]

for i, folds in enumerate(BATCHES, 1):
    out = '/content/out_b34_batch' + str(i) + '.csv'
    meta = '/content/out_b34_batch' + str(i) + '.run.json'
    if os.path.exists(os.path.join(OUT, os.path.basename(out))):
        print('batch', i, 'already in Drive, skipping')
        continue
    cmd = ('python scripts/train_v13_loso.py'
           ' --folds ' + folds +
           ' --epochs 15 --patience 3'
           ' --unfreeze 2 --finetune-epochs 5 --finetune-lr 1e-5'
           ' --keep-all-background --overwrite '
           + A +
           ' --out ' + out +
           ' --head-dir ' + HEADS +
           ' --run-metadata ' + meta)
    print()
    print(SEP, i, folds)
    print()
    r = subprocess.run(cmd, shell=True)
    if r.returncode != 0:
        print('batch', i, 'FAILED, stopping')
        break
    # Sync straight away. A session that dies during batch 4 should cost
    # batch 4, not batches 1 to 3.
    shutil.copy(out, os.path.join(OUT, os.path.basename(out)))
    shutil.copy(meta, os.path.join(OUT, os.path.basename(meta)))
    d = os.path.join(OUT, 'heads_b34_16')
    os.makedirs(d, exist_ok=True)
    for f in os.listdir(HEADS):
        shutil.copy(os.path.join(HEADS, f), os.path.join(d, f))
    print('batch', i, 'synced:', len(os.listdir(d)), 'head files in Drive')


In [ ]:
import glob, pandas as pd
fs = sorted(glob.glob('/content/out_b34_batch*.csv'))
if fs:
    t = pd.concat([pd.read_csv(f) for f in fs], ignore_index=True)
    t.to_csv('/content/out_b34_16.csv', index=False)
    shutil.copy('/content/out_b34_16.csv', os.path.join(OUT, 'out_b34_16.csv'))
    print(len(t), 'of 16 folds done')
    cols = ['station', 'loso_precision', 'loso_calls_retained', 'loso_fps_removed']
    print(t[cols].to_string(index=False))
    print()
    print('macro loso_precision', round(t.loso_precision.mean(), 4))
    print()
    print('Re-ranking figure, with each station own rejected detections in')
    print('training. NOT a held-out precision. The test that decides')
    print('anything is the scan, at each fold own fitted threshold.')
else:
    print('no batches completed')
